In [1]:
import pandas as pd

# Load your dataset (replace with your actual file path)
df = pd.read_csv("predictedemotionall.csv")

# List of target emotions
target_emotions = ["neutral", "anger", "disgust", "fear", "sadness", "surprise", "joy"]

# Filter rows where predicted_emotion is in the target list
filtered_df = df[df['predicted_emotion'].isin(target_emotions)]

# Save to a new CSV (optional)
filtered_df.to_csv("filteredemotionall.csv", index=False)

In [2]:
df['predicted_emotion'].value_counts()

predicted_emotion
anger                     2828
neutral                   1686
disgust                    492
surprise                   407
sadness                    265
fear                       138
Skipped - Invalid Text     121
joy                         20
confusion                    1
sad                          1
Name: count, dtype: int64

In [3]:
from datasets import load_dataset

dataset = load_dataset('csv', data_files='filteredemotionall.csv')  # columns: text, emotion
dataset = dataset['train'].train_test_split(test_size=0.2)

Generating train split: 0 examples [00:00, ? examples/s]

In [4]:
label_names = list(set(dataset['train']['predicted_emotion']))
label2id = {label: i for i, label in enumerate(label_names)}
id2label = {i: label for label, i in label2id.items()}

def encode_labels(example):
    example['label'] = label2id[example['predicted_emotion']]
    return example

dataset = dataset.map(encode_labels)

Map:   0%|          | 0/4668 [00:00<?, ? examples/s]

Map:   0%|          | 0/1168 [00:00<?, ? examples/s]

In [5]:
!pip install peft

In [6]:
from peft import LoraConfig, get_peft_model
from transformers import AutoTokenizer, AutoModelForSequenceClassification


model_name = "michellejieli/emotion_text_classifier"  # or any transformer model

tokenizer = AutoTokenizer.from_pretrained(model_name)
# Load model
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=len(label2id),
    id2label=id2label,
    label2id=label2id
)

# Configure LoRA
lora_config = LoraConfig(
    r=8,                  # rank of update matrices
    lora_alpha=32,        # scaling factor
    target_modules=["query", "value"],  # layers to apply LoRA to
    lora_dropout=0.1,
    bias="none",
    task_type="SEQ_CLS"   # task type
)

# Apply LoRA
model = get_peft_model(model, lora_config)

model.print_trainable_parameters()


trainable params: 743,431 || all params: 82,867,214 || trainable%: 0.8971


In [7]:
def tokenize_function(example):
    return tokenizer(example["body"], padding="max_length", truncation=True)

tokenized_dataset = dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/4668 [00:00<?, ? examples/s]

Map:   0%|          | 0/1168 [00:00<?, ? examples/s]

In [8]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./opt2-emotion-model",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_dir="./logs",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
)


In [22]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./opt-emotion-model",
    eval_strategy="epoch",      # Evaluate after each epoch
    save_strategy="epoch",             # Save best each epoch
    logging_dir="./logs",
    learning_rate=2e-5,                # Lower LR for small data
    per_device_train_batch_size=8,     # Smaller batch => better generalization
    per_device_eval_batch_size=8,
    num_train_epochs=5,                # Slightly more epochs but watch for overfit
    weight_decay=0.05,                 # A bit stronger regularization
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",  # Define selection metric
    save_total_limit=2,                # Keep only top 2 checkpoints
    warmup_ratio=0.1,                  # Warmup to stabilize first few steps
    logging_steps=20,                  # Frequent logs
    report_to="none"                   # Disable wandb unless using it
)


In [9]:
from transformers import Trainer, TrainingArguments
import evaluate

accuracy = evaluate.load("accuracy")

def compute_metrics(p):
    predictions, labels = p
    preds = predictions.argmax(-1)
    return accuracy.compute(predictions=preds, references=labels)


In [10]:
from transformers import EarlyStoppingCallback

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset['train'],
    eval_dataset=tokenized_dataset['test'],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)


C:\Users\pbahulek22\AppData\Local\Temp\ipykernel_30148\4065507653.py:3: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [11]:
trainer.train()


C:\Users\pbahulek22\AppData\Local\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss,Accuracy
1,No log,0.983597,0.650685
2,1.171600,0.907186,0.671233
3,1.171600,0.888578,0.678082


C:\Users\pbahulek22\AppData\Local\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
C:\Users\pbahulek22\AppData\Local\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


TrainOutput(global_step=876, training_loss=1.0682347615559895, metrics={'train_runtime': 21962.5751, 'train_samples_per_second': 0.638, 'train_steps_per_second': 0.04, 'total_flos': 1887221479514112.0, 'train_loss': 1.0682347615559895, 'epoch': 3.0})

In [12]:
trainer.save_model("./opt2-emotion-model")
tokenizer.save_pretrained("./opt2-emotion-model")


('./opt2-emotion-model\\tokenizer_config.json',
 './opt2-emotion-model\\special_tokens_map.json',
 './opt2-emotion-model\\vocab.json',
 './opt2-emotion-model\\merges.txt',
 './opt2-emotion-model\\added_tokens.json',
 './opt2-emotion-model\\tokenizer.json')

In [13]:
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
import torch

# --------------------------
# 1. Load Locally Saved Fine-Tuned Model
# --------------------------
model_dir = "./opt2-emotion-model"

tokenizer = AutoTokenizer.from_pretrained(model_dir)
model = AutoModelForSequenceClassification.from_pretrained(model_dir)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Create pipeline for prediction
emotion_classifier = pipeline(
    "text-classification",
    model=model,
    tokenizer=tokenizer,
    device=0 if torch.cuda.is_available() else -1
)

# --------------------------
# 2. Load Unseen Text Data
# --------------------------
# Replace with your file path
df = pd.read_csv("ner.csv")  # Should have a column named "text"

# Drop rows with missing text
df = df.dropna(subset=['thread_text_translated'])

# --------------------------
# 3. Predict Emotions
# --------------------------
predictions = emotion_classifier(df['thread_text_translated'].tolist(), truncation=True)

# Add predicted labels to dataframe
df['predictedemotion'] = [pred['label'] for pred in predictions]

# --------------------------
# 4. Save Results
# --------------------------
df.to_csv("opt2_nokia_emotion_detection.csv", index=False)
df['predictedemotion'].value_counts()

Device set to use cpu


predictedemotion
disgust    1762
sadness    1356
neutral    1028
fear        176
anger        97
joy           2
Name: count, dtype: int64

In [14]:
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
import torch

# --------------------------
# 1. Load Locally Saved Fine-Tuned Model
# --------------------------
model_dir = "./opt2-emotion-model"

tokenizer = AutoTokenizer.from_pretrained(model_dir)
model = AutoModelForSequenceClassification.from_pretrained(model_dir)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Create pipeline for prediction
emotion_classifier = pipeline(
    "text-classification",
    model=model,
    tokenizer=tokenizer,
    device=0 if torch.cuda.is_available() else -1
)

# --------------------------
# 2. Load Unseen Text Data
# --------------------------
# Replace with your file path
df = pd.read_csv("canadaaspectsentiment.csv")  # Should have a column named "text"

# Drop rows with missing text
df = df.dropna(subset=['body'])

# --------------------------
# 3. Predict Emotions
# --------------------------
predictions = emotion_classifier(df['body'].tolist(), truncation=True)

# Add predicted labels to dataframe
df['predictedemotion'] = [pred['label'] for pred in predictions]

# --------------------------
# 4. Save Results
# --------------------------
df.to_csv("opt2_canada_emotion_detection.csv", index=False)
df['predictedemotion'].value_counts()

Device set to use cpu


predictedemotion
disgust     927
sadness     837
anger        82
neutral      76
fear         48
surprise      6
joy           1
Name: count, dtype: int64

In [15]:
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
import torch

# --------------------------
# 1. Load Locally Saved Fine-Tuned Model
# --------------------------
model_dir = "./opt2-emotion-model"

tokenizer = AutoTokenizer.from_pretrained(model_dir)
model = AutoModelForSequenceClassification.from_pretrained(model_dir)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Create pipeline for prediction
emotion_classifier = pipeline(
    "text-classification",
    model=model,
    tokenizer=tokenizer,
    device=0 if torch.cuda.is_available() else -1
)

# --------------------------
# 2. Load Unseen Text Data
# --------------------------
# Replace with your file path
df = pd.read_csv("flintaspectfinetunefinal.csv")  # Should have a column named "text"

# Drop rows with missing text
df = df.dropna(subset=['body'])

# --------------------------
# 3. Predict Emotions
# --------------------------

predictions = emotion_classifier(df['body'].tolist(), truncation=True)

# Add predicted labels to dataframe
df['predictedemotion'] = [pred['label'] for pred in predictions]

# --------------------------
# 4. Save Results
# --------------------------
df.to_csv("opt2_flint_emotion_detection.csv", index=False)
df['predictedemotion'].value_counts()

Device set to use cpu


predictedemotion
sadness     14028
disgust      6884
anger         732
neutral       723
fear          316
surprise       27
joy            14
Name: count, dtype: int64

In [16]:
model_dir = "./opt2-emotion-model"

tokenizer = AutoTokenizer.from_pretrained(model_dir)
model = AutoModelForSequenceClassification.from_pretrained(model_dir)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Create pipeline for prediction
emotion_classifier = pipeline(
    "text-classification",
    model=model,
    tokenizer=tokenizer,
    device=0 if torch.cuda.is_available() else -1
)


# --------------------------
# 2. Load Unseen Text Data
# --------------------------
# Replace with your file path
df = pd.read_csv("predictedemotionalleval.csv")  # Should have a column named "text"

# Drop rows with missing text
df = df.dropna(subset=['body'])

# --------------------------
# 3. Predict Emotions
# --------------------------
predictions = emotion_classifier(df['body'].tolist(), truncation=True)

# Add predicted labels to dataframe
df['predictedemotionft'] = [pred['label'] for pred in predictions]

# --------------------------
# 4. Save Results
# --------------------------
df.to_csv("opt2_eval_data.csv", index=False)
print("Predictions saved to eval_data.csv") 


Device set to use cpu


Predictions saved to eval_data.csv


In [17]:
df.rename(columns={'predicted_emotion': 'predicted_emotion_llm', 'emotion': 'emotionwithoutft'}, inplace=True)

In [18]:
df = df[df['predicted_emotion_llm'] != 'concern']

In [19]:
from transformers import pipeline
import pandas as pd
from tqdm import tqdm

# Initialize emotion classifier with truncation enabled
classifier = pipeline(
    "sentiment-analysis",
    model="michellejieli/emotion_text_classifier",
    truncation=True,        # truncate long texts safely
    padding=True,           # pad shorter texts
    max_length=512,         # enforce BERT max token limit
    batch_size=16           # process in small batches for efficiency
)

# Example: assume df is already loaded and has a 'body' column
# df = pd.read_csv("your_file.csv")

# Apply the classifier
tqdm.pandas(desc="Classifying emotions")

def get_emotion(text):
    if pd.isnull(text) or not str(text).strip():
        return None
    try:
        result = classifier(str(text))[0]
        return result["label"]
    except Exception as e:
        print(f"Error on text: {text[:60]}... -> {e}")
        return None

df["nofinetune_emotion"] = df["body"].progress_apply(get_emotion)

# View results
print(df.head())

Device set to use cpu
Classifying emotions: 100%|████████████████████████████████████████████████████████| 2979/2979 [03:10<00:00, 15.67it/s]


   Unnamed: 0.14  Unnamed: 0.13  Unnamed: 0.12  Unnamed: 0.11  Unnamed: 0.10  \
0              0            NaN            NaN            NaN            NaN   
1              1            NaN            NaN            NaN            NaN   
2              2            NaN            NaN            NaN            NaN   
3              3            NaN            NaN            NaN            NaN   
4              4            NaN            NaN            NaN            NaN   

   Unnamed: 0.9  Unnamed: 0.8  Unnamed: 0.7  Unnamed: 0.6  Unnamed: 0.5  ...  \
0           NaN           NaN           NaN           NaN           NaN  ...   
1           NaN           NaN           NaN           NaN           NaN  ...   
2           NaN           NaN           NaN           NaN           NaN  ...   
3           NaN           NaN           NaN           NaN           NaN  ...   
4           NaN           NaN           NaN           NaN           NaN  ...   

           author  score  url         

In [20]:
import pandas as pd
from sklearn.metrics import classification_report

# Load your dataframes
#df_pred = pd.read_csv("/content/predicted_output.csv")       # Contains predictedemotion
#df_true = pd.read_csv("/content/filteredemotion_dataset.csv")           # Contains predicted_emotion

# Merge on thread_text_translated
#merged_df = pd.merge(df_true, df_pred, on="thread_text_translated", how="inner")

# Get ground truth and predicted labels
y_true = df["predicted_emotion_llm"]
y_pred = df["nofinetune_emotion"]

# Print class-wise metrics
report = classification_report(y_true, y_pred, digits=3)
print(report)

              precision    recall  f1-score   support

       anger      0.932     0.149     0.258      1552
     disgust      0.140     0.174     0.155       224
        fear      0.409     0.134     0.202        67
         joy      0.099     0.900     0.178        10
     neutral      0.338     0.926     0.495       785
     sadness      0.373     0.175     0.238       143
    surprise      0.237     0.141     0.177       198

    accuracy                          0.359      2979
   macro avg      0.361     0.371     0.243      2979
weighted avg      0.628     0.359     0.305      2979



In [21]:
y_true = df["predicted_emotion_llm"]
y_pred = df["predictedemotionft"]

# Print class-wise metrics
report = classification_report(y_true, y_pred, digits=3)
print(report)

              precision    recall  f1-score   support

       anger      0.274     0.017     0.032      1552
     disgust      0.065     0.254     0.104       224
        fear      0.239     0.239     0.239        67
         joy      0.250     0.100     0.143        10
     neutral      0.231     0.061     0.097       785
     sadness      0.043     0.524     0.080       143
    surprise      0.500     0.005     0.010       198

    accuracy                          0.075      2979
   macro avg      0.229     0.172     0.101      2979
weighted avg      0.250     0.075     0.060      2979

